Incremental MERGE for service_interactions

### What we're changing
**Before:** Overwrite the whole Silver table every run
**After:** MERGE — insert new tickets, update only the resolution data on previously-open tickets

### Why this pattern (not SCD2)
Tickets are facts, not dimensions. You don't need history of every state change —
you need the most recent state. A ticket goes: opened → in progress → resolved.
We care about the resolved state, not all three rows.

### The protective MERGE pattern
The key statement:
```sql
WHEN MATCHED
   AND target.resolution_hours IS NULL
   AND source.resolution_hours IS NOT NULL
THEN UPDATE ...
```

Translation: "Only update an existing ticket's resolution data if it was previously
NULL (the ticket was open) AND the incoming row has actual resolution data."

This protects against:
- Partial updates accidentally NULL-ing out good data
- Reprocessing the same delta twice creating data corruption

In [0]:
#catalog+Imports
CATALOG = "cx"
SCHEMA_BRONZE = "cx_bronze"
SCHEMA_SILVER = "cx_silver"
DELTA_PATH = "/Volumes/cx/default/cx_project_raw_delta"  

from pyspark.sql.functions import (
    col, current_timestamp, lit, trim, to_timestamp, when
)
from pyspark.sql.types import IntegerType, DoubleType
from datetime import datetime

run_start = datetime.now()
run_id = run_start.strftime("%Y%m%d_%H%M%S")
print(f"Pipeline run started: {run_id}")

Pipeline run started: 20260604_202404


## Step 1 — Bronze delta ingestion

In [0]:
#Import delta CSV to Bronze
bronze_delta_table = f"{CATALOG}.{SCHEMA_BRONZE}.service_interactions_incoming"

bronze_delta = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{DELTA_PATH}/service_interactions_delta.csv")
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("service_interactions_delta.csv"))
    .withColumn("pipeline_run_id", lit(run_id))
)

bronze_delta.write.mode("overwrite").format("delta").saveAsTable(bronze_delta_table)
delta_count = bronze_delta.count()
print(f"Bronze delta ingested: {delta_count:,} rows")


Bronze delta ingested: 2,200 rows


## Step 2 — Clean the delta

Same DQ rules as the original Silver build.

In [0]:
#Apply DQ rules to the Delta
valid_customer_ids_df = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.customers_scd2") \
    .filter("is_current = true") \
    .select("customer_id")

delta_clean = (
    spark.table(bronze_delta_table)
    .withColumn("ticket_id", trim(col("ticket_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("ticket_open_date", to_timestamp(col("ticket_open_date")))
    .withColumn("channel", trim(col("channel")))
    .withColumn("issue_category", trim(col("issue_category")))
    .withColumn("priority", trim(col("priority")))
    .withColumn("first_response_minutes", col("first_response_minutes").cast(DoubleType()))
    .withColumn("resolution_hours", col("resolution_hours").cast(DoubleType()))
    .withColumn("escalated_flag", col("escalated_flag").cast(DoubleType()).cast(IntegerType()))
    .withColumn("csat_post_ticket", col("csat_post_ticket").cast(DoubleType()).cast(IntegerType()))
    # Same DQ rule as before: negative resolution_hours → NULL
    .withColumn(
        "resolution_hours",
        when(col("resolution_hours") < 0, None).otherwise(col("resolution_hours"))
    )
)

# Orphan flag - tickets referencing customer_ids not in current customer master
delta_with_orphan = (
    delta_clean.alias("d")
    .join(valid_customer_ids_df.alias("c"), col("d.customer_id") == col("c.customer_id"), "left")
    .withColumn("dq_flag_orphan", when(col("c.customer_id").isNull(), 1).otherwise(0))
    .select("d.*", "dq_flag_orphan")
)

delta_with_orphan.createOrReplaceTempView("delta_clean")
print(f"Delta cleaned: {delta_with_orphan.count():,} rows")

Delta cleaned: 2,200 rows


## Step 3 — The protective MERGE

Two conditions covered:
1. **New ticket** (`WHEN NOT MATCHED`) → INSERT
2. **Existing open ticket gets resolved** (`WHEN MATCHED AND ...`) → UPDATE selectively

Critical safeguard: we only UPDATE if `target.resolution_hours IS NULL AND source.resolution_hours IS NOT NULL`.
This prevents accidentally NULL-ing out resolved tickets if a partial re-feed comes through.

In [0]:
#Execute Merge
silver_table = f"{CATALOG}.{SCHEMA_SILVER}.service_interactions"

merge_result = spark.sql(f"""
    MERGE INTO {silver_table} AS target
    USING delta_clean AS source
    ON target.ticket_id = source.ticket_id

    -- Case 1: Existing open ticket now has resolution data
    WHEN MATCHED
        AND target.resolution_hours IS NULL
        AND source.resolution_hours IS NOT NULL
    THEN UPDATE SET
        target.resolution_hours = source.resolution_hours,
        target.csat_post_ticket = source.csat_post_ticket,
        target.escalated_flag = source.escalated_flag

    -- Case 2: Brand-new ticket
    WHEN NOT MATCHED THEN INSERT (
        ticket_id, customer_id, ticket_open_date, channel, issue_category,
        priority, first_response_minutes, resolution_hours, escalated_flag,
        csat_post_ticket, dq_flag_orphan
    ) VALUES (
        source.ticket_id, source.customer_id, source.ticket_open_date, source.channel,
        source.issue_category, source.priority, source.first_response_minutes,
        source.resolution_hours, source.escalated_flag, source.csat_post_ticket,
        source.dq_flag_orphan
    )
""")

print("MERGE complete")

MERGE complete


## Step 4 — Capture MERGE statistics

Delta tracks every operation. We can read the operation metrics from the table history
to see exactly how many rows were inserted vs updated.

In [0]:
#Pull stats from Delta history
history = spark.sql(f"DESCRIBE HISTORY {silver_table} LIMIT 1").collect()[0]
op_metrics = history["operationMetrics"]

rows_inserted = int(op_metrics.get("numTargetRowsInserted", 0))
rows_updated = int(op_metrics.get("numTargetRowsUpdated", 0))
rows_unchanged = delta_count - rows_inserted - rows_updated

print(f"MERGE stats from Delta history:")
print(f"  Rows inserted: {rows_inserted:,}")
print(f"  Rows updated:  {rows_updated:,}")
print(f"  Rows unchanged (filtered by WHEN MATCHED AND conditions): {rows_unchanged:,}")

MERGE stats from Delta history:
  Rows inserted: 0
  Rows updated:  0
  Rows unchanged (filtered by WHEN MATCHED AND conditions): 2,200


In [0]:
#log this run
run_end = datetime.now()
duration_sec = (run_end - run_start).total_seconds()

run_log_table = f"{CATALOG}.{SCHEMA_SILVER}.pipeline_run_log"

run_log_data = [(
    run_id,
    "service_interactions_merge",
    run_start,
    run_end,
    float(duration_sec),
    delta_count,
    rows_inserted,
    rows_updated,
    "SUCCESS",
)]

run_log_df = spark.createDataFrame(
    run_log_data,
    schema="run_id STRING, pipeline_name STRING, start_time TIMESTAMP, end_time TIMESTAMP, duration_seconds DOUBLE, rows_in_delta INT, current_rows INT, historical_rows INT, status STRING"
)

run_log_df.write.mode("append").format("delta").saveAsTable(run_log_table)
print(f"\nRun logged: {run_id}")


Run logged: 20260604_202404


## Step 5 — Verify the MERGE worked correctly

Look at a few of the tickets that should have been updated (previously NULL resolution_hours
that are now filled in). Pick a sample from the delta and trace it.

In [0]:
%sql
--Verify updated tickets now have resolution data
 -- Tickets that were updated should now have resolution_hours NOT NULL
 SELECT ticket_id, resolution_hours, csat_post_ticket
 FROM cx.cx_silver.service_interactions
 WHERE ticket_id IN (
   SELECT ticket_id FROM cx.cx_bronze.service_interactions_incoming
   WHERE resolution_hours IS NOT NULL
   LIMIT 5
 );

ticket_id,resolution_hours,csat_post_ticket
TKT_00400005,26.81,null
TKT_00400006,51.04,null
TKT_00400007,3.99,null
TKT_00400008,39.66,null
TKT_00400013,21.19,null


In [0]:
%sql
--Check pipeline run log
 SELECT * FROM cx.cx_silver.pipeline_run_log
 ORDER BY start_time DESC;

run_id,pipeline_name,start_time,end_time,duration_seconds,rows_in_delta,current_rows,historical_rows,status
20260604_202404,service_interactions_merge,2026-06-04T20:24:04.054Z,2026-06-04T20:24:21.565Z,17.510961,2200,0,0,SUCCESS
20260604_202301,service_interactions_merge,2026-06-04T20:23:01.296Z,2026-06-04T20:23:24.811Z,23.514796,2200,0,0,SUCCESS
20260604_201259,service_interactions_merge,2026-06-04T20:12:59.824Z,2026-06-04T20:20:51.270Z,471.445131,2200,2000,4,SUCCESS
20260604_190935,customers_scd2_merge,2026-06-04T19:09:35.867Z,2026-06-04T19:29:01.208Z,1165.341891,550,50500,41,SUCCESS
20260604_190935,customers_scd2_merge,2026-06-04T19:09:35.867Z,2026-06-04T19:21:22.655Z,706.787988,550,50500,41,SUCCESS


## Step 6 — Idempotency proof

Re-run cells 4–10 with the SAME delta data. Expected behavior:
- `rows_inserted` should drop to **0** (all those ticket_ids already exist)
- `rows_updated` should drop to **0** (resolution_hours is no longer NULL after the first run)

Run twice. Check the pipeline_run_log. The second run should show zero changes.

In [0]:
%sql
SELECT
  s.ticket_id,
  s.resolution_hours AS now_resolved,
  s.csat_post_ticket
FROM cx.cx_silver.service_interactions s
WHERE s.ticket_id LIKE 'TKT_0000%'  -- original ticket ID range, not new
  AND s.resolution_hours IS NOT NULL
  AND s.ticket_id IN (
    SELECT ticket_id FROM cx.cx_bronze.service_interactions_incoming
  )
LIMIT 10;

ticket_id,now_resolved,csat_post_ticket
TKT_00000178,203.11,3
TKT_00000397,16.85,4
TKT_00000773,6.74,1
TKT_00001003,2.36,null
TKT_00001384,91.91,null
TKT_00001411,20.37,4
TKT_00002133,13.67,5
TKT_00002135,11.36,4
TKT_00002498,3.99,3
TKT_00002997,0.62,4


In [0]:
%sql
SELECT
  timestamp,
  operationMetrics.numTargetRowsInserted AS inserted,
  operationMetrics.numTargetRowsUpdated AS updated
FROM (DESCRIBE HISTORY cx.cx_silver.service_interactions)
ORDER BY timestamp DESC
LIMIT 5;

timestamp,inserted,updated
2026-06-04T20:24:20.000Z,0,0
2026-06-04T20:23:21.000Z,0,0
2026-06-04T20:18:24.000Z,2000,4
2026-05-19T20:33:28.000Z,null,null
2026-05-19T19:56:44.000Z,null,null
